In [1]:
!pip install -q google-generativeai

In [2]:
!pip install -q chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

In [3]:
import google.generativeai as genai
from sentence_transformers import SentenceTransformer

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [4]:
import chromadb

print("ChromaDB installed")

ChromaDB installed


In [5]:
from google.colab import files

uploaded = files.upload()

Saving chroma_db.zip to chroma_db.zip


In [6]:
!unzip chroma_db.zip

Archive:  chroma_db.zip
   creating: chroma_db/
  inflating: chroma_db/chroma.sqlite3  
   creating: chroma_db/c8665e1f-6a15-454e-8e5b-338c20fc959e/
  inflating: chroma_db/c8665e1f-6a15-454e-8e5b-338c20fc959e/data_level0.bin  
  inflating: chroma_db/c8665e1f-6a15-454e-8e5b-338c20fc959e/index_metadata.pickle  
  inflating: chroma_db/c8665e1f-6a15-454e-8e5b-338c20fc959e/header.bin  
  inflating: chroma_db/c8665e1f-6a15-454e-8e5b-338c20fc959e/link_lists.bin  
  inflating: chroma_db/c8665e1f-6a15-454e-8e5b-338c20fc959e/length.bin  


In [7]:
embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

print("Embedding model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded


In [8]:
genai.configure(
    api_key="YOUR_API_KEY"
)

gemini_model = genai.GenerativeModel(
    "gemini-2.5-flash"
)

print("Gemini loaded")

Gemini loaded


In [9]:
import chromadb

client = chromadb.PersistentClient(
    path="./chroma_db"
)

collection = client.get_collection(
    "aviation_incidents"
)

print(collection.count())

16502


In [10]:

def search_incidents(query, top_k=5):

    query_embedding = embedding_model.encode([query])

    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k
    )

    return results["documents"][0]

In [14]:
def analyze_incidents(query):

    incidents = search_incidents(query)

    context = "\n\n".join(incidents)

    prompt = f"""
You are an aviation safety analyst.

ONLY use information from the retrieved incidents below.

If there is insufficient evidence in the incidents to support a conclusion,
explicitly state "Insufficient evidence from retrieved incidents."

User Query:
{query}

Retrieved Incidents:
{context}

Task:

Analyze the retrieved incidents and provide:

1. Common Hazards
2. Contributing Factors
3. Potential Outcomes
4. Safety Recommendations

Requirements:
- Use bullet points.
- Reference the incident number(s) supporting each point.
- Do not use outside aviation knowledge.
- Do not make assumptions beyond the retrieved incidents.
- If evidence is weak, state so clearly.

Format:

## Common Hazards
- Hazard (Supported by Incident X, Y)

## Contributing Factors
- Factor (Supported by Incident X, Y)

## Potential Outcomes
- Outcome (Supported by Incident X, Y)

## Safety Recommendations
- Recommendation (Based on Incident X, Y)
"""

    response = gemini_model.generate_content(prompt)

    return response.text

In [15]:
def aviation_analysis(query):

    incidents = search_incidents(query)

    analysis = analyze_incidents(query)

    return {
        "retrieved_incidents": incidents,
        "analysis": analysis
    }

In [16]:
result = aviation_analysis(
    "Aircraft taxiing in dense fog with communication delays"
)

In [17]:
for i, incident in enumerate(result["retrieved_incidents"]):
    print(f"\nINCIDENT {i+1}")
    print(incident[:500])


INCIDENT 1
The GPS signal lost integrity while taxiing into parking spot. On the taxi out on the next flight; the GPS took approximately 5 minutes to link with sufficient satellites. Similar occurred on the succeeding flights in/out of Boston.Cause: I do not know. 

INCIDENT 2
I was assigned a series of flights by Dispatch in aircraft. The first flight I flew it empty from ZZZ to ZZZ1 to pick up cargo. It was flown under Part 91. The weather in ZZZ was categorized as LIFR; ceiling of 200 FT and visibility of half a mile. I picked up ATIS; taxied to the exit section of the ramp and called ZZZ Ground. I received my IFR clearance and taxi instructions. I was instructed to taxi on '1' and hold short of Runway XX at '2'. I complied. Whilke taxiing on '1'; the windshield be

INCIDENT 3
Our flight was directed by Ramp to enter at Gate XX and proceed to Gate XY to taxi through into Gate XZ; West Holding Pad at ZZZ. It was nighttime; and visibility was very poor due to inadequate illumination 

In [18]:
print(result["analysis"])

## Common Hazards
*   Spatial disorientation and difficulty maintaining taxiway centerline in low visibility. (Supported by Incident 2, 3)
*   Collision with other aircraft due to proximity. (Supported by Incident 3)
*   Taxiway excursion, leading to aircraft going off the paved surface. (Supported by Incident 2, 5)
*   Degradation or loss of navigation system integrity (e.g., GPS). (Supported by Incident 1)

## Contributing Factors
*   **Poor visibility conditions:**
    *   Dense fog (LIFR, 1/2 mile visibility). (Supported by Incident 2)
    *   Inadequate illumination during nighttime operations. (Supported by Incident 3, 5)
    *   Misted or obscured windshield. (Supported by Incident 2)
    *   Limited taxiway pavement markings and uncleared taxiway surfaces (snow/ice). (Supported by Incident 5)
*   **Communication failures or lack of information:**
    *   Ground control (Ramp) unaware of other aircraft's occupied positions. (Supported by Incident 3)
*   **Pilot perception and ju